# Pré-processamento 2 — Maceió

Este notebook realiza:
1. Carregamento dos dados filtrados espacialmente
2. Geração da grade espacial e visualização
3. Atribuição de cada ponto de crime à célula correspondente do grid

**Entrada:** `filtered_maceio_inside_polygon.csv`, `osm_maceio.geojson`

**Saída:** `maceio_with_grid_index.csv`, `maceio_grids.html`

## 1. Carregamento dos dados

In [1]:
# Carrega o CSV de Maceió filtrado espacialmente (dentro do polígono)

import pandas as pd

df_crimes = pd.read_csv(
    "./output/maceio/filtered_maceio_inside_polygon.csv",
    delimiter=";",
)
# Tamanho do grid calculado no notebook pre-processing-1
# Fórmula: N = round(sqrt(A_bbox / 0.2)), onde 0.2 km² é a área-alvo por célula
GRID_SIZE = 73

In [2]:
## 2. Visualização dos dados
display(df_crimes)
df_crimes.info()

,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO
0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió
1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió
2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió
3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió
4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió
...,...,...,...,...,...
72644,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió
72645,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió
72646,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió
72647,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió


<class 'pandas.DataFrame'>
RangeIndex: 72649 entries, 0 to 72648
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   LONGITUDE    72649 non-null  float64
 1   LATITUDE     72649 non-null  float64
 2   ROUBO_GROUP  72649 non-null  str    
 3   DATA_HORA    72649 non-null  str    
 4   CIDADE_FATO  72649 non-null  str    
dtypes: float64(2), str(3)
memory usage: 2.8 MB


## 2. Geração do grid e atribuição de células

- Cria uma grade regular N×N sobre o bounding box de Maceió
- Visualiza o grid em mapa interativo (folium)
- Para cada ponto de crime, identifica a célula do grid correspondente (point-in-polygon)
- Utiliza processamento paralelo (pandarallel) para acelerar a atribuição

In [ ]:
# Geração do grid e atribuição de cada crime à sua célula
# generate_grid(): cria a grade N×N e visualiza em mapa
# find_polygon(): para cada ponto, encontra a célula que o contém
# Resultado: CSV com coluna 'grid_cell_id' indicando o índice da célula do grid

from shapely.geometry import Point
import shapely.geometry
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from pandarallel import pandarallel
import os

# Ensure at least 1 worker, handle case where os.cpu_count() returns None or 0
cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 20))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def generate_grid(grid_size):
    mun = gpd.read_file("./output/maceio/osm_maceio.geojson")
    mun = mun.to_crs("EPSG:4326")
    gdf = mun[mun["name"] == "Maceió"].copy()

    bbox = gdf.total_bounds
    min_lon, min_lat, max_lon, max_lat = (bbox[2], bbox[1], bbox[0], bbox[3])

    lon = np.linspace(min_lon, max_lon, grid_size + 1)
    lat = np.linspace(min_lat, max_lat, grid_size + 1)

    latlons = []
    for i in range(len(lat) - 1):
        for k in range(len(lon) - 1):
            latlons.append((lat[k], lon[i], lat[k + 1], lon[i + 1]))

    m = folium.Map(
        location=((min_lat + max_lat) / 2, (min_lon + max_lon) / 2), zoom_start=11
    )
    for k in latlons:
        folium.Rectangle(
            [(k[0], k[1]), (k[2], k[3])], color="red", fill="pink", fill_opcity=0.5
        ).add_to(m)
    cgeo = (
        gdf.set_crs("epsg:4326")
        .sample(1)
        .pipe(lambda d: d.to_crs(d.estimate_utm_crs()))["geometry"]
        # .centroid.buffer(10000)
        .to_crs("epsg:4326")
        .__geo_interface__
    )

    geo_j = folium.GeoJson(data=cgeo)
    geo_j.add_to(m)
    m.save("./output/maceio/maceio_grids.html")

    return (min_lon, min_lat, max_lon, max_lat, latlons)


def find_polygon(row, df):
    for p_idx, elem in df.iterrows():
        if elem["polygon"].contains(Point(row["LONGITUDE"], row["LATITUDE"])):
            polygon_num = p_idx
            break
    return pd.Series({"grid_cell_id": polygon_num})


# generate grid boxes
min_lon, min_lat, max_lon, max_lat, grid_bboxes = generate_grid(grid_size=GRID_SIZE)

# turn boxes into polygons and that list into a dataframe
polygon_list = []
for elem in grid_bboxes:
    # elem is (min_lat, min_lon, max_lat, max_lon)
    polygon = shapely.geometry.box(elem[1], elem[0], elem[3], elem[2], ccw=True)
    polygon_list.append(polygon)
df_poly = pd.DataFrame(polygon_list, columns=["polygon"])


print("Finding the grid cell for each point...")
df_final = df_crimes.merge(
    df_crimes.parallel_apply(find_polygon, df=df_poly, axis=1),
    left_index=True,
    right_index=True,
)

df_final.to_csv(f"./output/maceio/maceio_with_grid_index.csv", sep=";")
display(df_final.shape)

INFO: Pandarallel will run on 20 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Finding the grid cell for each point...
